In [5]:
import numpy as np
from Py6S import SixS, Geometry, AtmosProfile,AeroProfile, Wavelength, Altitudes, GroundReflectance

In [ ]:
from Py6S import SixS, Geometry, AtmosProfile, AeroProfile, Wavelength, Altitudes

def _parse_down_up_total(obj):
    s = str(obj)
    # 例: "Downward: 0.913330, Upward: 0.924020, Total: 0.843940"
    def grab(key):
        i = s.find(key)
        if i < 0:
            return None
        j = s.find(":", i)
        k = s.find(",", j)
        if k < 0:
            k = len(s)
        return float(s[j+1:k].strip())

    d = grab("Downward")
    u = grab("Upward")
    t = grab("Total")
    if (d is None) or (u is None) or (t is None):
        raise ValueError(f"Failed to parse Downward/Upward/Total from: {s}")
    return d, u, t

def _get_terms_from_outputs(outputs):
    rho_a = float(outputs.atmospheric_intrinsic_reflectance)

    s_sph = str(outputs.spherical_albedo)
    S = float(s_sph.split("Total:")[-1].strip())

    scat_d, scat_u, _ = _parse_down_up_total(outputs.transmittance_total_scattering)
    gas_d,  gas_u,  _ = _parse_down_up_total(outputs.transmittance_global_gas)

    T_s = scat_d * gas_d  # T(theta_s)
    T_v = scat_u * gas_u  # T(theta_v)

    return rho_a, T_s, T_v, S


def run_py6s_coeffs(
    wavelengths_um,
    sun_zenith, sun_azimuth,
    view_zenith, view_azimuth,
    aod550,
    water, ozone,
    target_alt_km=0.0,
    aerosol_type="continental",
):
    s = SixS()

    # geometry
    g = Geometry.User()
    g.solar_z = float(sun_zenith)
    g.solar_a = float(sun_azimuth)
    g.view_z  = float(view_zenith)
    g.view_a  = float(view_azimuth)
    s.geometry = g

    # atmosphere
    s.atmos_profile = AtmosProfile.UserWaterAndOzone(
        water=float(water),   # g/cm^2 ≒ cm PW
        ozone=float(ozone)    # atm-cm
    )

    # aerosol
    at = aerosol_type.lower()
    if at == "continental":
        s.aero_profile = AeroProfile.Continental
    elif at == "maritime":
        s.aero_profile = AeroProfile.Maritime
    elif at == "urban":
        s.aero_profile = AeroProfile.Urban
    elif at == "desert":
        s.aero_profile = AeroProfile.Desert
    else:
        raise ValueError(f"Unsupported aerosol type: {aerosol_type}")

    s.aot550 = float(aod550)

    # altitude
    s.altitudes = Altitudes()
    s.altitudes.set_sensor_satellite_level()
    s.altitudes.set_target_custom_altitude(float(target_alt_km))

    coeffs = []
    for wl in wavelengths_um:
        s.wavelength = Wavelength(float(wl))
        s.run()
        rho_a, T_s, T_v, S = _get_terms_from_outputs(s.outputs)
        coeffs.append((rho_a, T_s, T_v, S))
    return coeffs


In [ ]:
# center wavelengths in micrometers for four bands
wavelengths_um = [0.490, 0.566, 0.665, 0.865]

sun_zenith  = 55.82836
sun_azimuth = 162.9697
view_zenith = 4.94857
view_azimuth= 11.75590

aod550 = 0.013329965062439442
water  = 0.22773604094982147  # cm
ozone  = 0.2840748727321625   # atm-cm

coeffs = run_py6s_coeffs(
    wavelengths_um=wavelengths_um,
    sun_zenith=sun_zenith, sun_azimuth=sun_azimuth,
    view_zenith=view_zenith, view_azimuth=view_azimuth,
    aod550=aod550,
    water=water,
    ozone=ozone,
    target_alt_km=0.0,
    aerosol_type="continental",
)


In [10]:
for wl, (rho_a, T_s, T_v, S) in zip(wavelengths_um, coeffs):
    print(f"wl={wl:.3f} um  rho_a={rho_a:.6f}  T_s={T_s:.6f}  T_v={T_v:.6f}  S={S:.6f}")


wl=0.490 um  rho_a=0.066000  T_s=0.863290  T_v=0.919280  S=0.126190
wl=0.566 um  rho_a=0.035000  T_s=0.872332  T_v=0.925976  S=0.078110
wl=0.665 um  rho_a=0.019000  T_s=0.932387  T_v=0.961843  S=0.044430
wl=0.865 um  rho_a=0.007000  T_s=0.981831  T_v=0.990250  S=0.017790
